# PICON Tutorial

PICON (Persona Interrogation framework for Consistency evaluation)은 LLM 기반 페르소나 에이전트를 자동으로 인터뷰하고 세 가지 차원에서 평가합니다:

- **Internal Consistency**: 자기 모순 없이 일관된 답변을 하는가
- **External Consistency**: 웹 검색 기반으로 실제 사실과 부합하는가
- **Retest Stability**: 동일 질문을 반복했을 때 일관성 있게 답변하는가

## 1. 설치

In [ ]:
# PyPI에서 설치 (간단)
# !pip install picon-eval

# 개발 모드 설치 (전체 기능 — Character.AI, Google GenAI 등)
# !pip install -e ".[all]"

In [ ]:
import picon
print(picon.__version__)

## 2. 환경 변수 설정

API 키를 `.env` 파일에 저장하거나 아래 셀에서 직접 설정합니다.

In [ ]:
import os

# 방법 1: .env 파일 자동 로드 (python-dotenv 사용)
# from dotenv import load_dotenv
# load_dotenv()

# 방법 2: 직접 설정
os.environ["OPENAI_API_KEY"]  = "YOUR_OPENAI_KEY"          # gpt-* 모델 사용 시 필수
os.environ["GEMINI_API_KEY"]  = "YOUR_GEMINI_KEY"          # gemini/* 모델 사용 시 필수
os.environ["SERPER_API_KEY"]  = "YOUR_SERPER_KEY"          # 외부 검증(external) 시 필수
os.environ["GOOGLE_GEOCODE"]  = "YOUR_GOOGLE_GEOCODE_KEY"  # 주소 검증 시 필수

# 선택 사항
# os.environ["ANTHROPIC_API_KEY"]   = "YOUR_ANTHROPIC_KEY"
# os.environ["GOOGLE_CLAIM_SEARCH"] = "YOUR_GOOGLE_API_KEY"
# os.environ["GOOGLE_CX_ID"]        = "YOUR_CUSTOM_SEARCH_ENGINE_ID"

## 3. Simple API: `picon.run()`

가장 간단한 사용 방법입니다. 단 몇 줄로 인터뷰 + 평가를 모두 실행합니다.

In [ ]:
import picon

result = picon.run(
    model="gpt-5",
    persona="You are a 35-year-old software engineer living in Seoul.",
    name="John",
    num_turns=20,
    num_sessions=2,
    do_eval=True,
)

print(result.eval_scores)
result.save("results/john.json")

## 4. Component-Based Usage

각 에이전트를 직접 구성하여 파이프라인을 세밀하게 제어합니다.

In [ ]:
from picon import Questioner, EntityExtractor, Evaluator, Interviewee
from picon import InterrogationSimulation

questioner   = Questioner(model="gpt-5")
extractor    = EntityExtractor(model="gpt-5.1")
evaluator    = Evaluator(model="gemini/gemini-2.5-flash")
interviewee  = Interviewee(
    model="gpt-5",
    persona="You are a 35-year-old software engineer living in Seoul.",
    name="John",
)

sim = InterrogationSimulation(
    interviewee=interviewee,
    questioner=questioner,
    extractor=extractor,
    evaluator=evaluator,
    num_turns=20,
    num_sessions=2,
)
result = sim.run(do_eval=True)

print(result.eval_scores)
result.save("results/john_component.json")

### 최소 설정 (기본 모델 자동 사용)

In [ ]:
from picon import Interviewee, InterrogationSimulation

interviewee = Interviewee(model="gpt-5", persona="You are ...", name="John")
result = InterrogationSimulation(interviewee=interviewee, num_turns=20).run()

## 5. 외부 에이전트 엔드포인트 평가

이미 실행 중인 OpenAI 호환 엔드포인트(`/v1/chat/completions`)를 바로 평가합니다.

In [ ]:
from picon import Interviewee, InterrogationSimulation

interviewee = Interviewee(api_base="http://localhost:8000/v1", name="MyAgent")
result = InterrogationSimulation(interviewee=interviewee, num_turns=20).run()

## 6. Self-hosted Model (vLLM)

```bash
# 터미널에서 vLLM 서버를 먼저 실행하세요
vllm serve meta-llama/Llama-3-8B --port 8000
```

In [ ]:
from picon import Interviewee, InterrogationSimulation

interviewee = Interviewee(
    api_base="http://localhost:8000/v1",
    model="meta-llama/Llama-3-8B",
    persona="You are a 30-year-old teacher named Jane...",
    name="Jane",
)
result = InterrogationSimulation(interviewee=interviewee).run()

## 7. 인터뷰와 평가 분리 실행

In [ ]:
import picon

# Step 1: 인터뷰만 실행
interview_result = picon.run_interview(
    name="John",
    model="gpt-5",
    persona="You are a 35-year-old software engineer...",
    num_turns=20,
    num_sessions=2,
)

# Step 2: 평가 실행 (eval_factors: "internal", "external", "intra", "inter")
persona_stats = picon.run_evaluation(
    interview_result,
    eval_factors=["internal", "external"],
)
print(persona_stats)

## 8. 기존 결과 파일 평가

In [ ]:
import picon

scores = picon.evaluate(
    "results/john.json",
    eval_factors=["internal", "external"],
)
print(scores)

## 9. 커스텀 래핑 서버 (RAG, API 호출 등)

OpenAI 호환 엔드포인트가 없는 에이전트는 FastAPI로 래핑합니다.

In [ ]:
server_code = '''
import time
from fastapi import FastAPI, Request
import uvicorn

app = FastAPI()

def generate_response(messages: list) -> str:
    """여기에 자신만의 에이전트 로직을 구현하세요 (RAG, API 호출 등)."""
    user_message = messages[-1]["content"]
    return "This is my response."

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    content = generate_response(body.get("messages", []))
    return {
        "id": f"chatcmpl-{int(time.time())}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": "my-agent",
        "choices": [{"index": 0, "message": {"role": "assistant", "content": content}, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
    }

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8001)
'''

with open("server.py", "w") as f:
    f.write(server_code)

print("server.py 생성 완료. 터미널에서 `python server.py` 를 실행한 후 아래 셀을 실행하세요.")

In [ ]:
from picon import Interviewee, InterrogationSimulation

interviewee = Interviewee(api_base="http://localhost:8001/v1", name="MyCustomAgent")
result = InterrogationSimulation(interviewee=interviewee, num_turns=20).run(do_eval=True)
print(result.eval_scores)

## 10. 평가 지표 설명

| 지표 | 설명 |
|------|------|
| **Internal Responsiveness** | 질문에 대한 답변의 관련성 |
| **Internal Consistency** | 반복 질문에 대한 답변의 일관성 |
| **Internal Harmonic Mean** | 응답성과 일관성의 조화 평균 |
| **External Coverage** | 검증 가능한 주장을 포함한 턴의 비율 |
| **External Non-refutation Rate** | 웹 증거에 의해 반박되지 않은 주장의 비율 |
| **External Consistency (EC)** | Coverage와 Non-refutation Rate의 조화 평균 |
| **Retest Consistency (Inter)** | 세션 간 답변 안정성 |
| **Retest Consistency (Intra)** | 세션 내 답변 안정성 |

---

## 11. 논문 벤치마크: 8가지 페르소나 에이전트 평가

논문에서 사용된 8가지 페르소나 에이전트 유형을 Python에서 직접 실행하는 예제입니다.

| # | 에이전트 유형 | 데이터 소스 | 비고 |
|---|--------------|------------|------|
| 1 | **Nemotron** | `nvidia/Nemotron-Personas-*` (HuggingFace) | 7개 지역 데이터셋 |
| 2 | **Twin-2K-500** | `LLM-Digital-Twin/Twin-2K-500` (HuggingFace) | |
| 3 | **LLM-Generated** | `Tianyi-Lab/Personas` (HuggingFace) | 4가지 표현 방식 |
| 4 | **DeepPersona** | 로컬 JSON 프로파일 파일 | `DATASET_DIR` 필요 |
| 5 | **Human Simulacra** | 11명 RAG 기반 캐릭터 (로컬) | 래핑 서버 자동 기동 |
| 6 | **OpenCharacter** | `xywang1/OpenCharacter` (HuggingFace) | vLLM 서버 필요 |
| 7 | **Character.AI** | `picon/env/personas/characterai.json` | `CAI_TOKEN` 필요 |
| 8 | **ConsistentLLM** | `picon/env/personas/consistent_llm_personas.jsonl` | fine-tuned vLLM 필요 |

### 공통 설정

8개 에이전트 모두 동일한 PICON 파이프라인 설정을 사용합니다.

In [ ]:
# 논문 실험과 동일한 공통 설정
PICON_CONFIG = dict(
    questioner_model="gpt-5",
    extractor_model="gpt-5.1",
    web_search_model="gpt-5",
    evaluator_model="gemini/gemini-2.5-flash",
    nhd_model="gpt-5-nano",
    num_turns=50,
    num_sessions=2,
    do_eval=True,
)

# 샘플링 설정 (SAMPLE_N=0 이면 전체 실행)
SAMPLE_N = 10
SEED = 42

---

### 11-1. Nemotron

NVIDIA의 Nemotron-Personas 데이터셋 7개 지역(USA, Korea, Singapore, France, India, Japan, Brazil)에서 페르소나를 샘플링하여 평가합니다.

**필요 패키지**: `datasets`

**사용 서버**: `servers/nemotron_server.py` (내부적으로 LLM이 페르소나를 역할극)

In [ ]:
import json
import random
import subprocess
import time
import requests
import picon
from datasets import load_dataset
from picon.env.interviewee_simulator.persona_prompt_builders import build_nemotron_prompt

NEMOTRON_DATASETS = [
    ("nvidia/Nemotron-Personas-USA",       "usa",  "train"),
    ("nvidia/Nemotron-Personas-Korea",     "kor",  "train"),
    ("nvidia/Nemotron-Personas-Singapore", "sgp",  "train"),
    ("nvidia/Nemotron-Personas-France",    "fra",  "train"),
    ("nvidia/Nemotron-Personas-India",     "ind",  "en_IN"),
    ("nvidia/Nemotron-Personas-Japan",     "jpn",  "train"),
    ("nvidia/Nemotron-Personas-Brazil",    "bra",  "train"),
]
SIMULATOR_MODEL = "gemini/gemini-3-flash-preview"  # 페르소나 역할극에 사용할 모델
BASE_PORT = 8100

# 각 지역에서 최소 1개씩 샘플링, 나머지는 랜덤 배분
rng = random.Random(SEED)
n_groups = len(NEMOTRON_DATASETS)
quotas = [1] * n_groups
for _ in range(max(0, SAMPLE_N - n_groups)):
    quotas[rng.randrange(n_groups)] += 1

nemotron_personas = []
for (repo, region, split), quota in zip(NEMOTRON_DATASETS, quotas):
    ds = load_dataset(repo, split=split).shuffle(seed=SEED).select(range(min(quota, len(load_dataset(repo, split=split)))))
    for d in ds:
        uid = d.get("uuid", "unknown")
        nemotron_personas.append({
            "name":   f"Nemotron-{region.upper()}-{uid[:8]}",
            "prompt": build_nemotron_prompt(d),
        })

rng.shuffle(nemotron_personas)
print(f"로드된 Nemotron 페르소나 수: {len(nemotron_personas)}")
print(f"첫 번째 페르소나 이름: {nemotron_personas[0]['name']}")
print(f"페르소나 프롬프트 앞 200자:\n{nemotron_personas[0]['prompt'][:200]}...")

In [ ]:
import tempfile, os

nemotron_results = []

for i, persona in enumerate(nemotron_personas):
    port = BASE_PORT + i + 1
    print(f"[{i+1}/{len(nemotron_personas)}] {persona['name']} (port={port})")

    # 페르소나 프롬프트를 임시 파일에 저장
    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, prefix="nemotron_") as f:
        f.write(persona["prompt"])
        tmpfile = f.name

    # 래핑 서버 시작
    proc = subprocess.Popen([
        "python", "servers/nemotron_server.py",
        "--port", str(port),
        "--model", SIMULATOR_MODEL,
        "--persona_file", tmpfile,
        "--name", persona["name"],
    ])

    # 서버 준비 대기 (최대 30초)
    for _ in range(30):
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=1).ok:
                break
        except Exception:
            time.sleep(1)

    # PICON 평가 실행
    result = picon.run(
        api_base=f"http://localhost:{port}/v1",
        name=persona["name"],
        output_dir="data/results/nemotron",
        **PICON_CONFIG,
    )
    nemotron_results.append(result)

    proc.terminate()
    proc.wait()
    os.unlink(tmpfile)
    print(f"  평가 점수: {result.eval_scores}")

print(f"\nNemotron 완료: {len(nemotron_results)}개 페르소나")

---

### 11-2. Twin-2K-500

실제 사람의 디지털 트윈 데이터셋 `LLM-Digital-Twin/Twin-2K-500`에서 페르소나를 로드하여 평가합니다.

**필요 패키지**: `datasets`

**사용 서버**: `servers/twin_2k_500_server.py`

In [ ]:
import json
import random
import subprocess
import tempfile
import time
import os
import requests
import picon
from datasets import load_dataset
from picon.env.interviewee_simulator.persona_prompt_builders import build_twin_2k_500_prompt

TWIN_SIMULATOR_MODEL = "gemini/gemini-2.5-flash"
BASE_PORT = 8100

# 데이터셋 로드 및 샘플링
dataset = load_dataset("LLM-Digital-Twin/Twin-2K-500", "full_persona", split="data")
if SAMPLE_N > 0:
    dataset = dataset.shuffle(seed=SEED).select(range(min(SAMPLE_N, len(dataset))))

twin_personas = []
for data in dataset:
    twin_personas.append({
        "name":   f"Twin-{data['pid']}",
        "prompt": build_twin_2k_500_prompt(json.dumps(data["persona_json"])),
    })

print(f"로드된 Twin-2K-500 페르소나 수: {len(twin_personas)}")
print(f"첫 번째 페르소나: {twin_personas[0]['name']}")

In [ ]:
twin_results = []

for i, persona in enumerate(twin_personas):
    port = BASE_PORT + i + 1
    print(f"[{i+1}/{len(twin_personas)}] {persona['name']} (port={port})")

    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, prefix="twin_") as f:
        f.write(persona["prompt"])
        tmpfile = f.name

    proc = subprocess.Popen([
        "python", "servers/twin_2k_500_server.py",
        "--port", str(port),
        "--model", TWIN_SIMULATOR_MODEL,
        "--persona_file", tmpfile,
        "--name", persona["name"],
    ])

    for _ in range(30):
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=1).ok:
                break
        except Exception:
            time.sleep(1)

    result = picon.run(
        api_base=f"http://localhost:{port}/v1",
        name=persona["name"],
        output_dir="data/results/twin_2k_500",
        **PICON_CONFIG,
    )
    twin_results.append(result)

    proc.terminate()
    proc.wait()
    os.unlink(tmpfile)
    print(f"  평가 점수: {result.eval_scores}")

print(f"\nTwin-2K-500 완료: {len(twin_results)}개 페르소나")

---

### 11-3. LLM-Generated

`Tianyi-Lab/Personas` 데이터셋을 사용합니다. 페르소나 표현 방식은 4가지 중 선택 가능합니다:
- `descriptive` (기본): 서술형 페르소나
- `objective`: 객관적 테이블 형식
- `subjective`: 주관적 테이블 형식
- `meta`: 메타 페르소나

**사용 서버**: `servers/llm_generated_server.py`

In [ ]:
import json
import subprocess
import tempfile
import time
import os
import requests
import picon
from datasets import load_dataset
from picon.env.interviewee_simulator.persona_prompt_builders import (
    build_llm_generated_prompt,
    extract_llm_generated_name,
)

LLM_GEN_PERSONA_TYPE  = "descriptive"  # descriptive | objective | subjective | meta
LLM_GEN_SIMULATOR_MODEL = "gemini/gemini-3-flash-preview"
BASE_PORT = 8100

dataset = load_dataset("Tianyi-Lab/Personas", split="train")
if SAMPLE_N > 0:
    dataset = dataset.shuffle(seed=SEED).select(range(min(SAMPLE_N, len(dataset))))

# 사용 가능한 모델 prefix 자동 탐지
available_prefixes = [c[:-len("_descriptive_persona")] for c in dataset.column_names if c.endswith("_descriptive_persona")]
prefix = next((p for p in ["Llama-3.1-70B-Instruct"] if p in available_prefixes), available_prefixes[0])

llm_gen_personas = []
for data in dataset:
    raw = {
        "descriptive_persona":      data.get(f"{prefix}_descriptive_persona", ""),
        "objective_table_persona":  data.get(f"{prefix}_objective_table_persona", ""),
        "subjective_table_persona": data.get(f"{prefix}_subjective_table_persona", ""),
        "meta_persona":             data.get("meta_persona", ""),
    }
    persona_prompt = build_llm_generated_prompt(json.dumps(raw), persona_type=LLM_GEN_PERSONA_TYPE)
    name = extract_llm_generated_name(raw["descriptive_persona"]) or f"LLM-Persona-{data.get('persona_number', '0')}"
    llm_gen_personas.append({"name": name, "prompt": persona_prompt})

print(f"로드된 LLM-Generated 페르소나 수: {len(llm_gen_personas)} (type={LLM_GEN_PERSONA_TYPE})")
print(f"첫 번째 페르소나: {llm_gen_personas[0]['name']}")

In [ ]:
llm_gen_results = []

for i, persona in enumerate(llm_gen_personas):
    port = BASE_PORT + i + 1
    print(f"[{i+1}/{len(llm_gen_personas)}] {persona['name']} (port={port})")

    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, prefix="llmgen_") as f:
        f.write(persona["prompt"])
        tmpfile = f.name

    proc = subprocess.Popen([
        "python", "servers/llm_generated_server.py",
        "--port", str(port),
        "--model", LLM_GEN_SIMULATOR_MODEL,
        "--persona_file", tmpfile,
        "--name", persona["name"],
    ])

    for _ in range(30):
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=1).ok:
                break
        except Exception:
            time.sleep(1)

    result = picon.run(
        api_base=f"http://localhost:{port}/v1",
        name=persona["name"],
        output_dir="data/results/llm_generated",
        **PICON_CONFIG,
    )
    llm_gen_results.append(result)

    proc.terminate()
    proc.wait()
    os.unlink(tmpfile)
    print(f"  평가 점수: {result.eval_scores}")

print(f"\nLLM-Generated 완료: {len(llm_gen_results)}개 페르소나")

---

### 11-4. DeepPersona

로컬 JSON 프로파일 파일들에서 페르소나를 로드합니다. `DATASET_DIR`을 DeepPersona 데이터셋 경로로 설정하세요.

**사전 조건**: `DATASET_DIR` 환경변수 또는 아래 변수에 데이터셋 경로 설정

**사용 서버**: `servers/deeppersona_server.py`

In [ ]:
import glob
import json
import random
import subprocess
import tempfile
import time
import os
import requests
import picon
from picon.env.interviewee_simulator.persona_prompt_builders import build_deeppersona_prompt

DATASET_DIR = os.environ.get("DATASET_DIR", "/path/to/deeppersona")  # 실제 경로로 변경
DEEPPERSONA_SIMULATOR_MODEL = "gemini/gemini-2.5-flash"
BASE_PORT = 8100

if not os.path.isdir(DATASET_DIR):
    raise FileNotFoundError(f"DATASET_DIR 경로가 존재하지 않습니다: {DATASET_DIR}")

# 모든 (파일, 프로필키) 쌍 수집 후 샘플링
files = sorted(glob.glob(os.path.join(DATASET_DIR, "*.json")))
pairs = []
for f in files:
    data = json.load(open(f))
    for key in data.keys():
        pairs.append((f, key))

rng = random.Random(SEED)
if SAMPLE_N > 0:
    pairs = rng.sample(pairs, min(SAMPLE_N, len(pairs)))

deeppersona_personas = []
for filepath, profile_key in pairs:
    data = json.load(open(filepath))
    profile = data.get(profile_key, data)
    deeppersona_personas.append({
        "name":   profile_key,
        "prompt": build_deeppersona_prompt(profile),
    })

print(f"로드된 DeepPersona 페르소나 수: {len(deeppersona_personas)}")

In [ ]:
deeppersona_results = []

for i, persona in enumerate(deeppersona_personas):
    port = BASE_PORT + i + 1
    print(f"[{i+1}/{len(deeppersona_personas)}] {persona['name']} (port={port})")

    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, prefix="deeppersona_") as f:
        f.write(persona["prompt"])
        tmpfile = f.name

    proc = subprocess.Popen([
        "python", "servers/deeppersona_server.py",
        "--port", str(port),
        "--model", DEEPPERSONA_SIMULATOR_MODEL,
        "--persona_file", tmpfile,
        "--name", persona["name"],
    ])

    for _ in range(30):
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=1).ok:
                break
        except Exception:
            time.sleep(1)

    result = picon.run(
        api_base=f"http://localhost:{port}/v1",
        name=persona["name"],
        output_dir="data/results/deeppersona",
        **PICON_CONFIG,
    )
    deeppersona_results.append(result)

    proc.terminate()
    proc.wait()
    os.unlink(tmpfile)
    print(f"  평가 점수: {result.eval_scores}")

print(f"\nDeepPersona 완료: {len(deeppersona_results)}개 페르소나")

---

### 11-5. Human Simulacra

11명의 고정 캐릭터를 RAG 기반으로 시뮬레이션합니다. 래핑 서버(`servers/human_simulacra_server.py`)가 자동으로 기동됩니다.

> RAG 인덱스 로딩 시간이 길기 때문에 서버 대기 시간을 **60초**로 설정합니다.

In [ ]:
import random
import subprocess
import time
import requests
import picon

ALL_CHARACTERS = [
    "Mary Jones",
    "Haley Collins",
    "Sara Ochoa",
    "James Jones",
    "Tami Clark",
    "Michael Miller",
    "Kevin Kelly",
    "Erica Walker",
    "Leslie Nichols",
    "Robert Scott",
    "Marsh Zhaleh",
]
HS_SIMULATOR_MODEL = "gemini/gemini-2.5-flash"
BASE_PORT = 8100

rng = random.Random(SEED)
characters = rng.sample(ALL_CHARACTERS, min(SAMPLE_N, len(ALL_CHARACTERS))) if SAMPLE_N > 0 else ALL_CHARACTERS

print(f"평가할 Human Simulacra 캐릭터 ({len(characters)}명): {characters}")

In [ ]:
hs_results = []

for i, character_name in enumerate(characters):
    port = BASE_PORT + i + 1
    print(f"[{i+1}/{len(characters)}] {character_name} (port={port})")

    proc = subprocess.Popen([
        "python", "servers/human_simulacra_server.py",
        "--port", str(port),
        "--character_name", character_name,
        "--model", HS_SIMULATOR_MODEL,
    ])

    # RAG 로딩 시간 고려하여 최대 60초 대기
    for _ in range(60):
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=1).ok:
                break
        except Exception:
            time.sleep(1)

    result = picon.run(
        api_base=f"http://localhost:{port}/v1",
        name=character_name,
        output_dir="data/results/human_simulacra",
        **PICON_CONFIG,
    )
    hs_results.append(result)

    proc.terminate()
    proc.wait()
    print(f"  평가 점수: {result.eval_scores}")

print(f"\nHuman Simulacra 완료: {len(hs_results)}명")

---

### 11-6. OpenCharacter

`xywang1/OpenCharacter` 데이터셋에서 캐릭터를 로드하여 vLLM으로 서빙된 OpenCharacter-SFT 모델로 평가합니다.

**사전 조건**: vLLM 서버를 먼저 실행해야 합니다.
```bash
vllm serve <your-opencharacter-model> --port 8000
```

**사용 서버**: `servers/opencharacter_server.py`

In [ ]:
import re
import json
import subprocess
import tempfile
import time
import os
import requests
import picon
from datasets import load_dataset
from picon.env.interviewee_simulator.persona_prompt_builders import build_opencharacter_prompt

# vLLM 서버 설정 — 실제 서빙 중인 주소와 모델명으로 변경
VLLM_BASE  = "http://localhost:8000/v1"
VLLM_MODEL = "anonymous/opencharacter-sft-llama-3-8b-instruct"
BASE_PORT  = 8100

dataset = load_dataset("xywang1/OpenCharacter", "Synthetic-Character", split="train")
if SAMPLE_N > 0:
    dataset = dataset.shuffle(seed=SEED).select(range(min(SAMPLE_N, len(dataset))))

openchar_personas = []
for data in dataset:
    name_match = re.match(r"Name:\s(.*)\n", data["character"])
    if not name_match:
        continue
    name = name_match.group(1).strip()
    prompt = build_opencharacter_prompt(data["persona"], data["character"])
    openchar_personas.append({"name": name, "prompt": prompt})

print(f"로드된 OpenCharacter 페르소나 수: {len(openchar_personas)}")
print(f"vLLM 엔드포인트: {VLLM_BASE} / {VLLM_MODEL}")

In [ ]:
openchar_results = []

for i, persona in enumerate(openchar_personas):
    port = BASE_PORT + i + 1
    print(f"[{i+1}/{len(openchar_personas)}] {persona['name']} (port={port})")

    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, prefix="openchar_") as f:
        f.write(persona["prompt"])
        tmpfile = f.name

    proc = subprocess.Popen([
        "python", "servers/opencharacter_server.py",
        "--port", str(port),
        "--vllm_base", VLLM_BASE,
        "--vllm_model", VLLM_MODEL,
        "--persona_file", tmpfile,
        "--name", persona["name"],
    ])

    for _ in range(30):
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=1).ok:
                break
        except Exception:
            time.sleep(1)

    result = picon.run(
        api_base=f"http://localhost:{port}/v1",
        name=persona["name"],
        output_dir="data/results/opencharacter",
        **PICON_CONFIG,
    )
    openchar_results.append(result)

    proc.terminate()
    proc.wait()
    os.unlink(tmpfile)
    print(f"  평가 점수: {result.eval_scores}")

print(f"\nOpenCharacter 완료: {len(openchar_results)}개 페르소나")

---

### 11-7. Character.AI

Character.AI의 실제 캐릭터들을 평가합니다. 캐릭터 목록은 `picon/env/personas/characterai.json`에 정의되어 있습니다.

**사전 조건**: Character.AI 세션 토큰(`CAI_TOKEN`)이 필요합니다.

**사용 서버**: `servers/characterai_server.py`

In [ ]:
import json
import random
import subprocess
import time
import os
import requests
import picon

CAI_TOKEN      = os.environ.get("CAI_TOKEN", "")  # .env에 CAI_TOKEN 설정 권장
PERSONAS_FILE  = "picon/env/personas/characterai.json"
BASE_PORT      = 8100

if not CAI_TOKEN:
    raise ValueError("CAI_TOKEN이 설정되지 않았습니다. os.environ['CAI_TOKEN'] = '...' 으로 설정하세요.")

personas = json.load(open(PERSONAS_FILE))
rng = random.Random(SEED)
if SAMPLE_N > 0:
    personas = rng.sample(personas, min(SAMPLE_N, len(personas)))

print(f"평가할 Character.AI 캐릭터 수: {len(personas)}")
for p in personas:
    print(f"  - {p['character_name']} (id={p['character_id']})")

In [ ]:
cai_results = []

for i, persona in enumerate(personas):
    port = BASE_PORT + i + 1
    name = persona["character_name"]
    cid  = persona["character_id"]
    print(f"[{i+1}/{len(personas)}] {name} (character_id={cid}, port={port})")

    proc = subprocess.Popen([
        "python", "servers/characterai_server.py",
        "--port", str(port),
        "--character_id", cid,
        "--user_id", CAI_TOKEN,
    ])

    # Character.AI 연결 초기화 대기
    for _ in range(30):
        try:
            if requests.get(f"http://localhost:{port}/", timeout=1).ok:
                break
        except Exception:
            time.sleep(1)

    result = picon.run(
        api_base=f"http://localhost:{port}/v1",
        name=name,
        output_dir="data/results/characterai",
        **PICON_CONFIG,
    )
    cai_results.append(result)

    proc.terminate()
    proc.wait()
    print(f"  평가 점수: {result.eval_scores}")

print(f"\nCharacter.AI 완료: {len(cai_results)}개 캐릭터")

---

### 11-8. ConsistentLLM

ConsistentLLM fine-tuned 모델을 평가합니다. vLLM으로 모델을 먼저 서빙해야 합니다.

**사전 조건**: vLLM 서버를 먼저 실행해야 합니다.
```bash
vllm serve anonymous/consistent_llm_llama-8b-sft-ppo-prompt --port 8001
```

**특이 사항**: 래핑 서버 없이 `--baseline_name consistent_llm` 플래그로 직접 PICON CLI에 페르소나 JSON을 전달합니다.

In [ ]:
import json
import random
import picon
from picon import Interviewee, InterrogationSimulation

PERSONAS_FILE      = "picon/env/personas/consistent_llm_personas.jsonl"
SIMULATOR_HOST     = "localhost"
SIMULATOR_PORT     = 8001
SIMULATOR_MODEL    = "hosted_vllm/anonymous/consistent_llm_llama-8b-sft-ppo-prompt"

# 페르소나 로드 및 샘플링
with open(PERSONAS_FILE) as f:
    all_personas = [json.loads(line) for line in f if line.strip()]

rng = random.Random(SEED)
if SAMPLE_N > 0:
    all_personas = rng.sample(all_personas, min(SAMPLE_N, len(all_personas)))

print(f"로드된 ConsistentLLM 페르소나 수: {len(all_personas)}")
print(f"Simulator: {SIMULATOR_MODEL} @ {SIMULATOR_HOST}:{SIMULATOR_PORT}")

In [ ]:
# ConsistentLLM은 picon Python API에서 simulator 파라미터를 직접 지원합니다
# (내부적으로 --baseline_name consistent_llm 로직과 동일)
consistent_llm_results = []

for i, persona_data in enumerate(all_personas):
    name = persona_data.get("name", f"ConsistentLLM-{i}")
    print(f"[{i+1}/{len(all_personas)}] {name}")

    interviewee = Interviewee(
        api_base=f"http://{SIMULATOR_HOST}:{SIMULATOR_PORT}/v1",
        model=SIMULATOR_MODEL,
        persona=json.dumps(persona_data, ensure_ascii=False),
        name=name,
    )

    from picon import Questioner, EntityExtractor, Evaluator
    sim = InterrogationSimulation(
        interviewee=interviewee,
        questioner=Questioner(model=PICON_CONFIG["questioner_model"]),
        extractor=EntityExtractor(model=PICON_CONFIG["extractor_model"]),
        evaluator=Evaluator(model=PICON_CONFIG["evaluator_model"]),
        num_turns=PICON_CONFIG["num_turns"],
        num_sessions=PICON_CONFIG["num_sessions"],
        output_dir="data/results/consistent_llm",
    )
    result = sim.run(do_eval=PICON_CONFIG["do_eval"])
    consistent_llm_results.append(result)
    print(f"  평가 점수: {result.eval_scores}")

print(f"\nConsistentLLM 완료: {len(consistent_llm_results)}개 페르소나")

---

## 12. 전체 결과 취합 및 비교

8가지 에이전트의 평가 결과를 하나의 테이블로 비교합니다.

In [ ]:
import statistics

def avg_scores(results: list, label: str) -> dict:
    """결과 리스트에서 각 지표의 평균을 계산합니다."""
    if not results:
        return {"agent": label}
    keys = [k for k in results[0].eval_scores.keys() if results[0].eval_scores[k] is not None]
    row = {"agent": label, "n": len(results)}
    for k in keys:
        vals = [r.eval_scores[k] for r in results if r.eval_scores.get(k) is not None]
        row[k] = round(statistics.mean(vals), 4) if vals else None
    return row

# 각 에이전트 결과를 딕셔너리로 수집
# (실행한 에이전트 결과만 포함하면 됩니다)
all_agent_results = {
    "Nemotron":       nemotron_results       if 'nemotron_results'       in dir() else [],
    "Twin-2K-500":    twin_results           if 'twin_results'           in dir() else [],
    "LLM-Generated":  llm_gen_results        if 'llm_gen_results'        in dir() else [],
    "DeepPersona":    deeppersona_results    if 'deeppersona_results'    in dir() else [],
    "HumanSimulacra": hs_results             if 'hs_results'             in dir() else [],
    "OpenCharacter":  openchar_results       if 'openchar_results'       in dir() else [],
    "Character.AI":   cai_results            if 'cai_results'            in dir() else [],
    "ConsistentLLM":  consistent_llm_results if 'consistent_llm_results' in dir() else [],
}

summary = [avg_scores(v, k) for k, v in all_agent_results.items() if v]

# 결과 출력
if summary:
    try:
        import pandas as pd
        df = pd.DataFrame(summary).set_index("agent")
        display(df)
    except ImportError:
        for row in summary:
            print(row)
else:
    print("아직 실행된 결과가 없습니다.")

In [ ]:
# 결과를 JSON 파일로 저장
import json
import os

os.makedirs("data/evaluation", exist_ok=True)
with open("data/evaluation/benchmark_summary.json", "w") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("벤치마크 결과 저장 완료: data/evaluation/benchmark_summary.json")